In [1]:
pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [optuna]


In [2]:
import numpy as np
import pandas as pd
import optuna
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import f1_score

In [3]:
# Importing datasets
X_train_trans = pd.read_csv('/content/drive/MyDrive/Deep Learning/Brain tumor MRI Detection Project /Dataset/X_train_trans.csv')
X_test_trans = pd.read_csv('/content/drive/MyDrive/Deep Learning/Brain tumor MRI Detection Project /Dataset/X_test_trans.csv')
y_train = pd.read_csv('/content/drive/MyDrive/Deep Learning/Brain tumor MRI Detection Project /Dataset/y_train.csv')
y_train = y_train.squeeze()
y_test = pd.read_csv('/content/drive/MyDrive/Deep Learning/Brain tumor MRI Detection Project /Dataset/y_test.csv')
y_test = y_test.squeeze()

In [4]:
# Checking dataset shape
print('shape of X_train_trans : ', X_train_trans.shape)
print('shape of X_test_trans :', X_test_trans.shape)
print('shape of y_train :', y_train.shape)
print('shape of y_test : ', y_test.shape)

shape of X_train_trans :  (5600, 1764)
shape of X_test_trans : (1600, 1764)
shape of y_train : (5600,)
shape of y_test :  (1600,)


In [16]:
def objective(trail):
  # Hyperparameters
  hidden_units = trail.suggest_int('hidden_units', 64, 512)
  dropout_rate = trail.suggest_float('dropout_rate',0.1, 0.5)
  learning_rate = trail.suggest_float('learning_rate', 1e-4, 1e-2, log = True)
  optimizer_name = trail. suggest_categorical('optimizer', ['adam', 'sgd'])

  # Model Defnition
  model = keras.Sequential([
                            layers.Input(shape = (1764, )),
                            layers.Dense(hidden_units, activation='relu'),
                            layers.Dropout(dropout_rate),
                            layers.Dense(4, activation='softmax')
                          ])

  if optimizer_name == 'sgd':
    optimizer = getattr(keras.optimizers, optimizer_name.upper())(learning_rate = learning_rate)
  else:
    optimizer = getattr(keras.optimizers, optimizer_name.title())(learning_rate = learning_rate)


  # model compile
  model.compile(
                 optimizer = optimizer,
                 loss='sparse_categorical_crossentropy',
                 metrics = ['accuracy']
               )

  # Training
  history = model.fit(
                       X_train_trans,
                       y_train,
                       validation_data = (X_test_trans, y_test),
                       batch_size = 128,
                       epochs = 100,
                       verbose =0
                      )

  # Return best validation f1_macro score
  return max(history.history['val_accuracy'])

In [7]:
study = optuna.create_study(direction="maximize")
study.optimize(objective, n_trials=50)
print("Best trial:")
trial = study.best_trial
print(f" Accuracy: {trial.value}")
for key, value in trial.params.items():
 print(f"{key}: {value}")

[I 2026-11-03 09:33:16,294] A new study created in memory with name: no-name-69f9de84-b526-494d-90cc-18befdb9db14
[I 2026-11-03 09:33:53,030] Trial 0 finished with value: 0.7493749856948853 and parameters: {'hidden_units': 187, 'dropout_rate': 0.15101024000378283, 'learning_rate': 0.00048466139471006037, 'optimizer': 'sgd'}. Best is trial 0 with value: 0.7493749856948853.
[I 2026-11-03 09:34:25,292] Trial 1 finished with value: 0.8756250143051147 and parameters: {'hidden_units': 208, 'dropout_rate': 0.2321904462424935, 'learning_rate': 0.0001820194216042684, 'optimizer': 'adam'}. Best is trial 1 with value: 0.8756250143051147.
[I 2026-11-03 09:34:55,614] Trial 2 finished with value: 0.7975000143051147 and parameters: {'hidden_units': 449, 'dropout_rate': 0.2341267132775268, 'learning_rate': 0.0008768846589511794, 'optimizer': 'sgd'}. Best is trial 1 with value: 0.8756250143051147.
[I 2026-11-03 09:35:28,848] Trial 3 finished with value: 0.7681249976158142 and parameters: {'hidden_units

Best trial:
 Accuracy: 0.8887500166893005
hidden_units: 468
dropout_rate: 0.1649331465382886
learning_rate: 0.005794544000095293
optimizer: adam


In [8]:
best_params = study.best_trial.params

hidden_units = best_params['hidden_units']
dropout_rate = best_params['dropout_rate']
learning_rate = best_params['learning_rate']
optimizer_name = best_params['optimizer']

print("Best Hyperparameters:", best_params)


Best Hyperparameters: {'hidden_units': 468, 'dropout_rate': 0.1649331465382886, 'learning_rate': 0.005794544000095293, 'optimizer': 'adam'}


# Build ANN model

In [9]:
model = keras.Sequential([
    layers.Input(shape=(1764,)),

    layers.Dense(hidden_units, activation='relu'),
    layers.Dropout(dropout_rate),

    layers.Dense(4, activation='softmax')
])

# Select optimizer
if optimizer_name == 'adam':
    optimizer = keras.optimizers.Adam(
        learning_rate=learning_rate
    )
else:
    optimizer = keras.optimizers.SGD(
        learning_rate=learning_rate
    )

# Compile the model
model.compile(
    optimizer=optimizer,
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Display summary
model.summary()

Model: "sequential_50"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_100 (Dense)               │ (None, 468)            │       826,020 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_50 (Dropout)            │ (None, 468)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_101 (Dense)               │ (None, 4)              │         1,876 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 827,896 (3.16 MB)

 Trainable params: 827,896 (3.16 MB)

 Non-trainable params: 0 (0.00 B)

**Train the ANN Model**

In [10]:
history = model.fit(
                    X_train_trans,
                    y_train,
                    validation_data=(X_test_trans, y_test),
                    batch_size=128,
                    epochs=100,
                    verbose=1
                  )

Epoch 1/100
44/44 ━━━━━━━━━━━━━━━━━━━━ 3s 43ms/step - accuracy: 0.7529 - loss: 1.7836 - val_accuracy: 0.7875 - val_loss: 1.5944
Epoch 2/100
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.8948 - loss: 0.3647 - val_accuracy: 0.8300 - val_loss: 1.4394
Epoch 3/100
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9571 - loss: 0.1275 - val_accuracy: 0.8331 - val_loss: 1.6787
Epoch 4/100
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 6ms/step - accuracy: 0.9570 - loss: 0.1341 - val_accuracy: 0.8494 - val_loss: 1.6262
Epoch 5/100
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9739 - loss: 0.0847 - val_accuracy: 0.8519 - val_loss: 2.0270
Epoch 6/100
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9757 - loss: 0.0741 - val_accuracy: 0.8456 - val_loss: 2.1558
Epoch 7/100
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9666 - loss: 0.1302 - val_accuracy: 0.8194 - val_loss: 2.9661
Epoch 8/100
44/44 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.9405 - loss: 0.3057 - val_accuracy: 0.8087 - 

**Evaluate the Model**

In [11]:
test_loss, test_accuracy = model.evaluate(
                                          X_test_trans,
                                          y_test,
                                          verbose=0
                                         )

print(f'Test Loss: {test_loss:.4f}')
print(f'Test Accuracy: {test_accuracy:.4f}')
print(f'Test Accuracy (%): {test_accuracy * 100:.2f}%')

Test Loss: 76.1131
Test Accuracy: 0.8619
Test Accuracy (%): 86.19%


**Make predictions**

In [12]:
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix

# Predict class probabilities
y_pred_prob = model.predict(X_test_trans)

# Convert probabilities into predicted class labels
y_pred = np.argmax(y_pred_prob, axis=1)



50/50 ━━━━━━━━━━━━━━━━━━━━ 0s 2ms/step


In [13]:
# Classification report
print(classification_report(y_test, y_pred))



              precision    recall  f1-score   support

           0       0.85      0.68      0.76       400
           1       0.78      0.85      0.82       400
           2       0.89      0.99      0.94       400
           3       0.93      0.92      0.92       400

    accuracy                           0.86      1600
   macro avg       0.86      0.86      0.86      1600
weighted avg       0.86      0.86      0.86      1600



In [14]:
# Confusion matrix
print(confusion_matrix(y_test, y_pred))

[[273  70  45  12]
 [ 40 341   4  15]
 [  1   1 397   1]
 [  6  24   2 368]]


In [15]:
# Save the trained ANN model
model.save('model.keras')

print("Model saved successfully!")

Model saved successfully!
